# Datawell Consultancy
## Project: Healthcare Claims Analytics
### Phase 1 — Data Quality, Cleaning & Master Dataset
---
**Prepared by:** Datawell Consultancy
**Version:** 1.0

---

## Problem Statement

A healthcare insurance company operates four disconnected data systems — claims records, provider profiles, patient demographics, and payment settlements. These systems have never been formally audited or connected. The business cannot answer basic operational questions such as:

- What is our overall claim approval and rejection rate?
- Which providers have the highest fraud risk?
- How long does it take to settle payments after claim approval?
- Which patient segments and diagnosis categories drive the highest claim volumes?

**Engagement objective:** Profile, clean, and unify all four data sources into a single analytics layer that answers these business questions reliably.

---

## Dataset Overview

| File | Records | Description |
|------|---------|-------------|
| `claims.csv` | 10,000 | Core claims table — amounts, status, fraud flag, diagnosis |
| `providers.csv` | 200 | Provider and hospital master data |
| `patients.csv` | 10,000 | Patient demographics and health history |
| `payments.csv` | 10,000 | Payment settlement records per claim |

## Step 0: Environment Setup

In [1]:
import os
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Base directory setup
BASE_DIR         = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
RAW_DATA_DIR     = os.path.join(BASE_DIR, 'data', 'raw')
CLEANED_DATA_DIR = os.path.join(BASE_DIR, 'data', 'cleaned')
DASHBOARD_DIR    = os.path.join(BASE_DIR, 'dashboard')
DOCS_DIR         = os.path.join(BASE_DIR, 'docs')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Chart settings
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Environment ready')

Environment ready


## Step 1: Load Raw Data

In [2]:
claims_raw    = pd.read_csv(os.path.join(RAW_DATA_DIR, 'claims.csv'))
providers_raw = pd.read_csv(os.path.join(RAW_DATA_DIR, 'providers.csv'))
patients_raw  = pd.read_csv(os.path.join(RAW_DATA_DIR, 'patients.csv'))
payments_raw  = pd.read_csv(os.path.join(RAW_DATA_DIR, 'payments.csv'))

print('All files loaded successfully')
print(f'Claims    : {claims_raw.shape[0]:,} rows x {claims_raw.shape[1]} columns')
print(f'Providers : {providers_raw.shape[0]:,} rows x {providers_raw.shape[1]} columns')
print(f'Patients  : {patients_raw.shape[0]:,} rows x {patients_raw.shape[1]} columns')
print(f'Payments  : {payments_raw.shape[0]:,} rows x {payments_raw.shape[1]} columns')

All files loaded successfully
Claims    : 10,000 rows x 16 columns
Providers : 200 rows x 11 columns
Patients  : 10,000 rows x 12 columns
Payments  : 10,000 rows x 9 columns


## Step 2: Data Quality Audit

Structured audit across four dimensions:
1. Completeness — missing values
2. Uniqueness — duplicate records
3. Validity — data types and formats
4. Consistency — business rule compliance

In [3]:
def data_quality_report(df, name):
    """Generate a structured data quality report for any dataframe."""
    print(f'\n{"="*60}')
    print(f'  DATA QUALITY REPORT — {name.upper()}')
    print(f'{"="*60}')

    total_rows = len(df)
    print(f'\nShape      : {total_rows:,} rows x {len(df.columns)} columns')
    print(f'Duplicates : {df.duplicated().sum():,} rows ({df.duplicated().sum()/total_rows*100:.1f}%)')

    null_counts = df.isnull().sum()
    null_pct    = (null_counts / total_rows * 100).round(2)

    quality_df = pd.DataFrame({
        'Column'        : df.columns,
        'Data Type'     : df.dtypes.values,
        'Null Count'    : null_counts.values,
        'Null %'        : null_pct.values,
        'Unique Values' : [df[col].nunique() for col in df.columns]
    })

    print(f'\nColumn Profile:')
    print(quality_df.to_string(index=False))
    return quality_df

claims_quality    = data_quality_report(claims_raw,    'Claims')
providers_quality = data_quality_report(providers_raw, 'Providers')
patients_quality  = data_quality_report(patients_raw,  'Patients')
payments_quality  = data_quality_report(payments_raw,  'Payments')


  DATA QUALITY REPORT — CLAIMS

Shape      : 10,000 rows x 16 columns
Duplicates : 0 rows (0.0%)

Column Profile:
                               Column Data Type  Null Count  Null %  Unique Values
                             Claim_ID    object           0    0.00          10000
                          Provider_ID    object           0    0.00            200
                           Patient_ID    object           0    0.00           6290
                       Diagnosis_Code    object           0    0.00              8
                       Procedure_Code     int64           0    0.00              8
                         Claim_Amount   float64           0    0.00           9992
                      Approved_Amount   float64           0    0.00           9994
                       Insurance_Type    object           0    0.00              4
                Claim_Submission_Date    object           0    0.00           1820
                         Claim_Status    object        

## Step 3: Business Rules Validation

In [4]:
print('BUSINESS RULES VALIDATION')
print('='*60)

print('\nCLAIMS TABLE:')
print(f'  Claim amount <= 0                   : {(claims_raw["Claim_Amount"] <= 0).sum()}')
print(f'  Approved > Claimed (invalid)         : {(claims_raw["Approved_Amount"] > claims_raw["Claim_Amount"]).sum()}')
print(f'  Fraud flagged claims                 : {(claims_raw["Is_Fraud"] == 1).sum():,}')
print(f'  Fraud rate                           : {(claims_raw["Is_Fraud"].mean()*100):.2f}%')
print(f'  Claim status distribution            :')
for status, count in claims_raw['Claim_Status'].value_counts().items():
    print(f'    {status:<15}: {count:,} ({count/len(claims_raw)*100:.1f}%)')

print('\nPROVIDERS TABLE:')
print(f'  High risk providers                  : {(providers_raw["Is_High_Risk"] == 1).sum()}')
print(f'  Lapsed accreditation                 : {(providers_raw["Accreditation_Status"] == "Lapsed").sum()}')
print(f'  Rating below 2.0                     : {(providers_raw["Provider_Rating"] < 2.0).sum()}')

print('\nPATIENTS TABLE:')
print(f'  Age below 18                         : {(patients_raw["Patient_Age"] < 18).sum()}')
print(f'  Age above 90                         : {(patients_raw["Patient_Age"] > 90).sum()}')
print(f'  BMI below 10 (unrealistic)           : {(patients_raw["BMI"] < 10).sum()}')
print(f'  Chronic condition patients           : {(patients_raw["Chronic_Condition_Flag"] == 1).sum():,}')

print('\nPAYMENTS TABLE:')
print(f'  Payment amount <= 0                  : {(payments_raw["Payment_Amount"] <= 0).sum()}')
print(f'  Late payments                        : {(payments_raw["Late_Payment_Flag"] == 1).sum():,}')
print(f'  Failed or disputed payments          : {payments_raw["Payment_Status"].isin(["Failed", "Disputed"]).sum():,}')
print(f'  Payment status distribution          :')
for status, count in payments_raw['Payment_Status'].value_counts().items():
    print(f'    {status:<15}: {count:,} ({count/len(payments_raw)*100:.1f}%)')

BUSINESS RULES VALIDATION

CLAIMS TABLE:
  Claim amount <= 0                   : 0
  Approved > Claimed (invalid)         : 0
  Fraud flagged claims                 : 994
  Fraud rate                           : 9.94%
  Claim status distribution            :
    Approved       : 6,649 (66.5%)
    Rejected       : 1,702 (17.0%)
    Pending        : 1,649 (16.5%)

PROVIDERS TABLE:
  High risk providers                  : 43
  Lapsed accreditation                 : 43
  Rating below 2.0                     : 44

PATIENTS TABLE:
  Age below 18                         : 0
  Age above 90                         : 0
  BMI below 10 (unrealistic)           : 0
  Chronic condition patients           : 3,359

PAYMENTS TABLE:
  Payment amount <= 0                  : 0
  Late payments                        : 2,529
  Failed or disputed payments          : 3,320
  Payment status distribution          :
    Paid           : 5,071 (50.7%)
    Disputed       : 1,695 (17.0%)
    Failed         : 1,625 (

## Step 4: Statistical Summary

In [5]:
print('STATISTICAL SUMMARY — CLAIMS')
print(claims_raw[['Claim_Amount', 'Approved_Amount', 'Length_of_Stay',
                   'Days_Between_Service_and_Claim',
                   'Number_of_Claims_Per_Provider_Monthly']].describe().round(2).to_string())

print('\nSTATISTICAL SUMMARY — PAYMENTS')
print(payments_raw[['Payment_Amount', 'Days_to_Payment']].describe().round(2).to_string())

print('\nSTATISTICAL SUMMARY — PATIENTS')
print(patients_raw[['Patient_Age', 'BMI', 'Prior_Visits_12m']].describe().round(2).to_string())

print('\nSTATISTICAL SUMMARY — PROVIDERS')
print(providers_raw[['Years_in_Practice', 'Provider_Rating', 'Total_Beds']].describe().round(2).to_string())

STATISTICAL SUMMARY — CLAIMS
       Claim_Amount  Approved_Amount  Length_of_Stay  Days_Between_Service_and_Claim  Number_of_Claims_Per_Provider_Monthly
count     10,000.00        10,000.00       10,000.00                       10,000.00                              10,000.00
mean      25,152.28        21,409.13           14.97                           30.36                                  80.18
std       14,472.17        12,556.02            8.99                           17.25                                  40.62
min          104.65            78.34            0.00                            1.00                                  10.00
25%       12,682.02        10,783.52            7.00                           15.00                                  45.00
50%       25,069.08        21,224.88           15.00                           30.00                                  81.00
75%       37,850.59        31,781.20           23.00                           45.00                   

## Step 5: Data Cleaning

In [6]:
# ─────────────────────────────────────────
# CLEAN CLAIMS TABLE
# ─────────────────────────────────────────
claims = claims_raw.copy()

# Parse date
claims['Claim_Submission_Date'] = pd.to_datetime(claims['Claim_Submission_Date'])

# Date parts
claims['Claim_Year']  = claims['Claim_Submission_Date'].dt.year
claims['Claim_Month'] = claims['Claim_Submission_Date'].dt.month
claims['Claim_Quarter'] = claims['Claim_Submission_Date'].dt.quarter

# Approval ratio
claims['Approval_Ratio'] = (claims['Approved_Amount'] / claims['Claim_Amount']).round(4)

# Denial amount
claims['Denied_Amount'] = (claims['Claim_Amount'] - claims['Approved_Amount']).round(2)

# Claim amount buckets
claims['Claim_Amount_Bucket'] = pd.cut(
    claims['Claim_Amount'],
    bins=[0, 1000, 5000, 10000, 25000, float('inf')],
    labels=['Low (<1K)', 'Mid (1K-5K)', 'High (5K-10K)', 'Major (10K-25K)', 'Premium (25K+)']
)

# Approval flag
claims['Is_Approved'] = (claims['Claim_Status'] == 'Approved').astype(int)
claims['Is_Rejected'] = (claims['Claim_Status'] == 'Rejected').astype(int)
claims['Is_Pending']  = (claims['Claim_Status'] == 'Pending').astype(int)

print(f'Claims cleaned    : {len(claims):,} records')
print(f'New columns added : Claim_Year, Claim_Month, Claim_Quarter, Approval_Ratio, Denied_Amount, Claim_Amount_Bucket, Is_Approved, Is_Rejected, Is_Pending')
claims.head(3)

Claims cleaned    : 10,000 records
New columns added : Claim_Year, Claim_Month, Claim_Quarter, Approval_Ratio, Denied_Amount, Claim_Amount_Bucket, Is_Approved, Is_Rejected, Is_Pending


,Claim_ID,Provider_ID,Patient_ID,Diagnosis_Code,Procedure_Code,Claim_Amount,Approved_Amount,Insurance_Type,Claim_Submission_Date,Claim_Status,Visit_Type,Length_of_Stay,Days_Between_Service_and_Claim,Is_Fraud,Number_of_Claims_Per_Provider_Monthly,Chronic_Condition_Flag,Claim_Year,Claim_Month,Claim_Quarter,Approval_Ratio,Denied_Amount,Claim_Amount_Bucket,Is_Approved,Is_Rejected,Is_Pending
0,CLM000001,P003,PAT07010,M54.5,71046,"18,650.73","18,320.99",Medicare,2022-01-15,Approved,Inpatient,30,21,1,98,1,2022,1,1,0.98,329.74,Major (10K-25K),1,0,0
1,CLM000002,P023,PAT07249,K21.0,93000,"33,792.46","29,841.05",Private,2024-10-06,Approved,Outpatient,0,36,0,33,0,2024,10,4,0.88,"3,951.41",Premium (25K+),1,0,0
2,CLM000003,P187,PAT02739,Z00.00,27447,"31,882.43","23,503.82",Private,2022-09-13,Approved,Outpatient,6,40,0,18,1,2022,9,3,0.74,"8,378.61",Premium (25K+),1,0,0


In [7]:
# ─────────────────────────────────────────
# CLEAN PROVIDERS TABLE
# ─────────────────────────────────────────
providers = providers_raw.copy()

# Rating tier
providers['Rating_Tier'] = pd.cut(
    providers['Provider_Rating'],
    bins=[0, 2.0, 3.0, 4.0, 5.0],
    labels=['Poor', 'Average', 'Good', 'Excellent']
)

# Experience tier
providers['Experience_Tier'] = pd.cut(
    providers['Years_in_Practice'],
    bins=[0, 5, 15, 25, float('inf')],
    labels=['Junior', 'Mid', 'Senior', 'Expert']
)

# Hospital size
providers['Hospital_Size'] = pd.cut(
    providers['Total_Beds'],
    bins=[0, 100, 300, 600, float('inf')],
    labels=['Small', 'Medium', 'Large', 'Major']
)

print(f'Providers cleaned : {len(providers):,} records')
print(f'New columns added : Rating_Tier, Experience_Tier, Hospital_Size')
providers.head(3)

Providers cleaned : 200 records
New columns added : Rating_Tier, Experience_Tier, Hospital_Size


,Provider_ID,Provider_Name,Hospital_Name,Hospital_Type,Provider_Specialty,State,Years_in_Practice,Provider_Rating,Total_Beds,Accreditation_Status,Is_High_Risk,Rating_Tier,Experience_Tier,Hospital_Size
0,P001,Mr. Kevin Schultz DDS,"Golden, Smith and Macias",Government,Cardiology,FL,31,2.60,621,Accredited,1,Average,Expert,Major
1,P002,Kristi Webb,Thornton and Sons,Government,Pediatrics,FL,17,3.10,437,Lapsed,0,Good,Senior,Large
2,P003,Thomas Calderon,"Bradley, Boyle and Vargas",Non-Profit,Oncology,GA,10,4.40,134,Lapsed,1,Excellent,Mid,Medium


In [8]:
# ─────────────────────────────────────────
# CLEAN PATIENTS TABLE
# ─────────────────────────────────────────
patients = patients_raw.copy()

# Age group
patients['Age_Group'] = pd.cut(
    patients['Patient_Age'],
    bins=[0, 25, 35, 45, 55, 65, 100],
    labels=['18-25', '26-35', '36-45', '46-55', '56-65', '65+']
)

# BMI category
patients['BMI_Category'] = pd.cut(
    patients['BMI'],
    bins=[0, 18.5, 25.0, 30.0, float('inf')],
    labels=['Underweight', 'Normal', 'Overweight', 'Obese']
)

# High utiliser flag — more than 8 visits in 12 months
patients['High_Utiliser'] = (patients['Prior_Visits_12m'] > 8).astype(int)

print(f'Patients cleaned  : {len(patients):,} records')
print(f'New columns added : Age_Group, BMI_Category, High_Utiliser')
patients.head(3)

Patients cleaned  : 10,000 records
New columns added : Age_Group, BMI_Category, High_Utiliser


,Patient_ID,Patient_Name,Patient_Age,Patient_Gender,BMI,Smoking_Status,Employment_Status,Annual_Income_Band,Chronic_Condition_Flag,Prior_Visits_12m,State,Blood_Type,Age_Group,BMI_Category,High_Utiliser
0,PAT00001,Jonathan Smith,73,Male,35.40,No,Retired,Lower-Mid,1,15,GA,B+,65+,Obese,1
1,PAT00002,Jordan Mcintosh,76,Female,26.20,No,Employed,High,0,10,TX,A-,65+,Overweight,1
2,PAT00003,Erin Williams,87,Male,42.30,No,Employed,Mid,1,3,NY,O+,65+,Obese,0


In [9]:
# ─────────────────────────────────────────
# CLEAN PAYMENTS TABLE
# ─────────────────────────────────────────
payments = payments_raw.copy()

# Parse date
payments['Payment_Date'] = pd.to_datetime(payments['Payment_Date'])

# Payment speed tier
payments['Payment_Speed'] = pd.cut(
    payments['Days_to_Payment'],
    bins=[0, 15, 30, 60, float('inf')],
    labels=['Fast (0-15d)', 'Normal (15-30d)', 'Slow (30-60d)', 'Very Slow (60d+)']
)

# Is payment completed
payments['Is_Paid']     = (payments['Payment_Status'] == 'Paid').astype(int)
payments['Is_Failed']   = (payments['Payment_Status'] == 'Failed').astype(int)
payments['Is_Disputed'] = (payments['Payment_Status'] == 'Disputed').astype(int)

print(f'Payments cleaned  : {len(payments):,} records')
print(f'New columns added : Payment_Speed, Is_Paid, Is_Failed, Is_Disputed')
payments.head(3)

Payments cleaned  : 10,000 records
New columns added : Payment_Speed, Is_Paid, Is_Failed, Is_Disputed


,Payment_ID,Claim_ID,Payment_Date,Payment_Amount,Payment_Method,Payment_Status,Days_to_Payment,Payment_Reference,Late_Payment_Flag,Payment_Speed,Is_Paid,Is_Failed,Is_Disputed
0,PAY000001,CLM000001,2022-10-21,"18,069.69",Bank Transfer,Paid,82,fe2219e6-fa68-4d96-9293-8ee08085332a,1,Very Slow (60d+),1,0,0
1,PAY000002,CLM000002,2022-09-03,"28,969.23",Bank Transfer,Failed,92,908575b9-e594-4e50-8983-991b5a315bda,0,Very Slow (60d+),0,1,0
2,PAY000003,CLM000003,2022-08-02,"23,060.65",Bank Transfer,Paid,44,6e108672-249c-47f8-9e61-e2105255129e,0,Slow (30-60d),1,0,0


## Step 6: Build Unified Master Dataset

In [10]:
master = duckdb.query("""
    SELECT
        -- Claim fields
        c.Claim_ID,
        c.Claim_Submission_Date,
        c.Claim_Year,
        c.Claim_Month,
        c.Claim_Quarter,
        c.Claim_Amount,
        c.Approved_Amount,
        c.Denied_Amount,
        c.Approval_Ratio,
        c.Claim_Amount_Bucket,
        c.Claim_Status,
        c.Is_Approved,
        c.Is_Rejected,
        c.Is_Pending,
        c.Is_Fraud,
        c.Insurance_Type,
        c.Visit_Type,
        c.Diagnosis_Code,
        c.Procedure_Code,
        c.Length_of_Stay,
        c.Days_Between_Service_and_Claim,
        c.Number_of_Claims_Per_Provider_Monthly,
        c.Chronic_Condition_Flag,

        -- Provider fields
        p.Provider_ID,
        p.Provider_Name,
        p.Hospital_Name,
        p.Hospital_Type,
        p.Provider_Specialty,
        p.State                 AS Provider_State,
        p.Years_in_Practice,
        p.Provider_Rating,
        p.Rating_Tier,
        p.Total_Beds,
        p.Hospital_Size,
        p.Accreditation_Status,
        p.Is_High_Risk          AS Provider_Is_High_Risk,

        -- Patient fields
        pt.Patient_ID,
        pt.Patient_Name,
        pt.Patient_Age,
        pt.Age_Group,
        pt.Patient_Gender,
        pt.BMI,
        pt.BMI_Category,
        pt.Smoking_Status,
        pt.Employment_Status,
        pt.Annual_Income_Band,
        pt.Prior_Visits_12m,
        pt.High_Utiliser,
        pt.State                AS Patient_State,
        pt.Blood_Type,

        -- Payment fields
        pay.Payment_ID,
        pay.Payment_Date,
        pay.Payment_Amount,
        pay.Payment_Method,
        pay.Payment_Status,
        pay.Payment_Speed,
        pay.Days_to_Payment,
        pay.Late_Payment_Flag,
        pay.Is_Paid,
        pay.Is_Failed,
        pay.Is_Disputed

    FROM claims c
    LEFT JOIN providers p  ON c.Provider_ID = p.Provider_ID
    LEFT JOIN patients  pt ON c.Patient_ID  = pt.Patient_ID
    LEFT JOIN payments  pay ON c.Claim_ID   = pay.Claim_ID
""").df()

print(f'Master dataset built  : {len(master):,} rows x {len(master.columns)} columns')
print(f'Join coverage providers : {master["Provider_ID"].notna().mean()*100:.1f}%')
print(f'Join coverage patients  : {master["Patient_ID"].notna().mean()*100:.1f}%')
print(f'Join coverage payments  : {master["Payment_ID"].notna().mean()*100:.1f}%')
master.head(3)

Master dataset built  : 10,000 rows x 61 columns
Join coverage providers : 100.0%
Join coverage patients  : 100.0%
Join coverage payments  : 100.0%


,Claim_ID,Claim_Submission_Date,Claim_Year,Claim_Month,Claim_Quarter,Claim_Amount,Approved_Amount,Denied_Amount,Approval_Ratio,Claim_Amount_Bucket,Claim_Status,Is_Approved,Is_Rejected,Is_Pending,Is_Fraud,Insurance_Type,Visit_Type,Diagnosis_Code,Procedure_Code,Length_of_Stay,Days_Between_Service_and_Claim,Number_of_Claims_Per_Provider_Monthly,Chronic_Condition_Flag,Provider_ID,Provider_Name,Hospital_Name,Hospital_Type,Provider_Specialty,Provider_State,Years_in_Practice,Provider_Rating,Rating_Tier,Total_Beds,Hospital_Size,Accreditation_Status,Provider_Is_High_Risk,Patient_ID,Patient_Name,Patient_Age,Age_Group,Patient_Gender,BMI,BMI_Category,Smoking_Status,Employment_Status,Annual_Income_Band,Prior_Visits_12m,High_Utiliser,Patient_State,Blood_Type,Payment_ID,Payment_Date,Payment_Amount,Payment_Method,Payment_Status,Payment_Speed,Days_to_Payment,Late_Payment_Flag,Is_Paid,Is_Failed,Is_Disputed
0,CLM000001,2022-01-15,2022,1,1,"18,650.73","18,320.99",329.74,0.98,Major (10K-25K),Approved,1,0,0,1,Medicare,Inpatient,M54.5,71046,30,21,98,1,P003,Thomas Calderon,"Bradley, Boyle and Vargas",Non-Profit,Oncology,GA,10,4.40,Excellent,134,Medium,Lapsed,1,PAT07010,Steven Clark,85,65+,Female,21.50,Normal,No,Self-Employed,Mid,3,0,GA,A-,PAY000001,2022-10-21,"18,069.69",Bank Transfer,Paid,Very Slow (60d+),82,1,1,0,0
1,CLM000002,2024-10-06,2024,10,4,"33,792.46","29,841.05","3,951.41",0.88,Premium (25K+),Approved,1,0,0,0,Private,Outpatient,K21.0,93000,0,36,33,0,P023,Rhonda Petersen,Mason-Ford,Clinic,Pediatrics,NY,27,2.60,Average,567,Large,Accredited,0,PAT07249,Stephen Cunningham,73,65+,Female,34.60,Obese,Yes,Retired,High,4,0,PA,AB+,PAY000002,2022-09-03,"28,969.23",Bank Transfer,Failed,Very Slow (60d+),92,0,0,1,0
2,CLM000003,2022-09-13,2022,9,3,"31,882.43","23,503.82","8,378.61",0.74,Premium (25K+),Approved,1,0,0,0,Private,Outpatient,Z00.00,27447,6,40,18,1,P187,John Woods,Turner-Osborne,Government,General Practice,NY,8,4.20,Excellent,109,Medium,Accredited,1,PAT02739,William Harris,79,65+,Female,25.90,Overweight,No,Retired,Upper-Mid,4,0,TX,O-,PAY000003,2022-08-02,"23,060.65",Bank Transfer,Paid,Slow (30-60d),44,0,1,0,0


## Step 7: Export Cleaned Data

In [11]:
claims.to_csv(os.path.join(CLEANED_DATA_DIR, 'claims_cleaned.csv'), index=False)
providers.to_csv(os.path.join(CLEANED_DATA_DIR, 'providers_cleaned.csv'), index=False)
patients.to_csv(os.path.join(CLEANED_DATA_DIR, 'patients_cleaned.csv'), index=False)
payments.to_csv(os.path.join(CLEANED_DATA_DIR, 'payments_cleaned.csv'), index=False)
master.to_csv(os.path.join(CLEANED_DATA_DIR, 'master_dataset.csv'), index=False)

print('All cleaned files exported to data/cleaned/')
print(f'  claims_cleaned.csv    : {len(claims):,} rows')
print(f'  providers_cleaned.csv : {len(providers):,} rows')
print(f'  patients_cleaned.csv  : {len(patients):,} rows')
print(f'  payments_cleaned.csv  : {len(payments):,} rows')
print(f'  master_dataset.csv    : {len(master):,} rows')

All cleaned files exported to data/cleaned/
  claims_cleaned.csv    : 10,000 rows
  providers_cleaned.csv : 200 rows
  patients_cleaned.csv  : 10,000 rows
  payments_cleaned.csv  : 10,000 rows
  master_dataset.csv    : 10,000 rows
